# Demo 12f — Attention

**Class 12 · T1–T2** (slides: `slides/12_neural_networks.md`)

* **T1:** scaled dot-product attention on three tokens by hand, checked against `keras.layers.MultiHeadAttention`; masks.
* **T2:** why the scores are divided by $\sqrt{d_k}$; positional encodings; self-attention is blind to order; the cost against the sequence length.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import time

import keras
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from faa import demo, neural, plotlib
from faa.demo import show, slide

demo.setup("12")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0

## Slide: Attention by hand (T1)

Three tokens with two numbers each, $X = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix}$. Queries, keys and values are the tokens themselves ($W_Q = W_K = W_V = I$, $d_k = 2$).
Row $i$ of the weights says how much token $i$ listens to each token.

In [ ]:
X = np.array([[1.0, 0], [0, 1], [1, 1]])
S = X @ X.T
S_scaled = S / np.sqrt(2)
A = neural.softmax(S_scaled)
out, A2 = neural.attention(X, X, X)
slide("Attention by hand")
print("scores Q K^T:\n", S)
print("scaled by 1/sqrt(2):\n", np.round(S_scaled, 3))
print("weights (softmax of each row):\n", np.round(A, 3), "\nrow sums:", A.sum(1))
print("output A V:\n", np.round(A @ X, 3))
print("faa.neural.attention agrees:", np.allclose(out, A @ X) and np.allclose(A, A2))
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL * 0.5, 1.15))
for a, m, t in zip(ax, (S, A), ("scores $QK^\\top$", "weights $A$")):
    a.imshow(m, cmap="Blues")
    a.set_title(t)
    a.set_xticks(range(3), ["1", "2", "3"])
    a.set_yticks(range(3), ["1", "2", "3"])
    for (i, j), v in np.ndenumerate(m):
        a.text(
            j,
            i,
            f"{v:.2f}" if m is A else f"{v:.0f}",
            ha="center",
            va="center",
            fontsize=7,
            color="white" if v > m.max() * 0.6 else "black",
        )
plotlib.save(f, "t1_attention_hand")

## Slide: Against Keras (T1)

One head of `MultiHeadAttention` without biases: the four weight arrays are $W_Q$, $W_K$, $W_V$, $W_O$; our function takes the same arrays.

In [ ]:
rng = np.random.default_rng(SEED)
Xr = rng.normal(size=(1, 5, 8)).astype("float32")
mha = keras.layers.MultiHeadAttention(1, 4, use_bias=False)
mha(Xr, Xr)
wq, wk, wv, wo = mha.get_weights()
Q, K, V = (Xr[0] @ w[:, 0, :] for w in (wq, wk, wv))
head_out, Aw = neural.attention(Q, K, V)
slide("Against Keras")
print("parameters of one head: 3 x (8 x 4) + 4 x 8 =", neural.n_params(mha))
print("max difference with Keras output:", float(np.abs(np.array(mha(Xr, Xr)[0]) - head_out @ wo[0]).max()))
_, kw = mha(Xr, Xr, return_attention_scores=True)
print("max difference with Keras weights:", float(np.abs(np.array(kw[0, 0]) - Aw).max()))

## Slide: Masks (T1)

A language model must not see the future: scores of positions $j > i$ get $-\infty$ before the softmax.

In [ ]:
n = 5
scores = rng.normal(size=(n, n))
Am = neural.softmax(np.where(neural.causal_mask(n), scores, -1e9))
slide("Causal mask")
print("lower-triangular mask:\n", neural.causal_mask(n).astype(int))
print("weights with the mask (rows sum to 1, zeros above the diagonal):\n", np.round(Am, 2))
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL * 0.5, 1.1))
for a, m, t in zip(ax, (neural.softmax(scores), Am), ("no mask", "causal mask")):
    a.imshow(m, cmap="Blues", vmin=0, vmax=1)
    a.set_title(t)
    a.set_xlabel("key $j$")
    a.set_ylabel("query $i$")
plotlib.save(f, "t1_mask")

## Slide: Why divide by $\sqrt{d_k}$ (T2)

With entries of $q$ and $k$ independent, mean 0 and variance 1, $q^\top k$ has variance $d_k$. Without scaling the softmax saturates and its gradient vanishes.

In [ ]:
rows = []
n_tok = 10
for dk in (4, 16, 64, 256, 1024):
    q = rng.normal(size=(2000, dk))
    k = rng.normal(size=(2000, n_tok, dk))
    sc = np.einsum("bd,bnd->bn", q, k)
    for name, s in (("unscaled", sc), ("scaled", sc / np.sqrt(dk))):
        p = neural.softmax(s)
        # gradient of the largest weight with respect to its score: p (1 - p)
        pm = p.max(1)
        rows.append(
            {
                "d_k": dk,
                "scores": name,
                "std of q.k": s.std(),
                "largest weight": pm.mean(),
                "mean p(1-p)": (pm * (1 - pm)).mean(),
            }
        )
slide("Why divide by sqrt(d_k)")
show(pl.DataFrame(rows))
qh = rng.normal(size=(4000, 256))
kh = rng.normal(size=(4000, 256))
sc_h = (qh * kh).sum(1)
f, ax = plotlib.fig("col", 1.4)
ax.hist(sc_h, bins=40, color=RED, alpha=0.7, density=True, label="$q \\cdot k$")
ax.hist(sc_h / 16, bins=40, color=BLUE, alpha=0.7, density=True, label="$q \\cdot k / \\sqrt{d_k}$")
ax.set_xlabel("score ($d_k = 256$)")
ax.set_ylabel("density")
ax.legend(frameon=False, fontsize=6)
plotlib.save(f, "t2_score_hist")
print(f"d_k = 256: std of q.k {sc_h.std():.2f}, scaled {(sc_h / 16).std():.2f}")
f, ax = plotlib.fig("col", 1.7)
for name, c in (("unscaled", RED), ("scaled", BLUE)):
    sub = [r for r in rows if r["scores"] == name]
    ax.semilogx([r["d_k"] for r in sub], [r["largest weight"] for r in sub], "o-", color=c, label=name)
ax.set_xlabel("$d_k$")
ax.set_ylabel("largest attention weight")
ax.legend(frameon=False)
plotlib.save(f, "t2_scaling")

## Slide: Self-attention does not know the order (T2)

Permute the tokens: the outputs are the same rows, permuted. Positions must be added to the input.

In [ ]:
Xp = rng.normal(size=(6, 8))
perm = rng.permutation(6)
o1, _ = neural.attention(Xp, Xp, Xp)
o2, _ = neural.attention(Xp[perm], Xp[perm], Xp[perm])
pe = neural.positional_encoding(6, 8)
o3, _ = neural.attention(Xp + pe, Xp + pe, Xp + pe)
o4, _ = neural.attention(Xp[perm] + pe, Xp[perm] + pe, Xp[perm] + pe)
slide("Self-attention is permutation equivariant")
print(f"without positions: max |attention(P X) - P attention(X)| = {np.abs(o2 - o1[perm]).max():.1e}")
print(f"with positions   : max |attention(P X + PE) - P attention(X + PE)| = {np.abs(o4 - o3[perm]).max():.2f}")

## Slide: Positional encoding (T2)

$PE_{p,2i} = \sin(p / 10000^{2i/d})$, $PE_{p,2i+1} = \cos(p / 10000^{2i/d})$: every position is a different pattern of waves, fast in the first columns and slow in the last.

In [ ]:
pe = neural.positional_encoding(50, 32)
sim = pe @ pe.T
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL * 0.78, 1.0))
ax[0].imshow(pe.T, cmap="RdBu_r", aspect="auto")
ax[0].set_xlabel("position $p$")
ax[0].set_ylabel("column")
ax[1].plot(sim[0], color=BLUE)
ax[1].set_xlabel("position $p$")
ax[1].set_ylabel("$PE_0 \\cdot PE_p$")
plotlib.save(f, "t2_positional")
slide("Positional encoding")
print(f"PE(0).PE(p): p=0 {sim[0, 0]:.1f}, p=1 {sim[0, 1]:.2f}, p=5 {sim[0, 5]:.2f}, p=20 {sim[0, 20]:.2f}")
print(f"distances only: PE(10).PE(15) = {sim[10, 15]:.3f} and PE(30).PE(35) = {sim[30, 35]:.3f}")

## Slide: Cost against the length (T2)

Attention builds an $n \times n$ matrix: $O(n^2 d)$ operations and $O(n^2)$ memory; one pass of a recurrent layer is $O(n\,d^2)$ but strictly sequential.
Time of one NumPy attention head ($d = 64$) for growing $n$.

In [ ]:
rows = []
d = 64
for n in (256, 512, 1024, 2048, 4096):
    Z = rng.normal(size=(n, d)).astype("float32")
    neural.attention(Z, Z, Z)  # warm-up
    t0 = time.perf_counter()
    for _ in range(5):
        neural.attention(Z, Z, Z)
    rows.append({"n": n, "score entries n^2": n * n, "seconds": (time.perf_counter() - t0) / 5})
slide("Cost of attention")
show(pl.DataFrame(rows), digits=4)
print(
    "time ratio when n doubles:",
    [round(rows[i + 1]["seconds"] / rows[i]["seconds"], 1) for i in range(len(rows) - 1)],
    "(typical, machine dependent)",
)
f, ax = plotlib.fig("col", 1.6)
ns = np.array([r["n"] for r in rows])
ax.loglog(ns, [r["seconds"] for r in rows], "o-", color=BLUE, label="measured")
ax.loglog(ns, rows[-1]["seconds"] * (ns / ns[-1]) ** 2, ls="--", color=GREY, label="$\\propto T^2$")
ax.set_xticks(ns, [str(n) for n in ns], fontsize=5.5, rotation=45)
ax.minorticks_off()
ax.set_xlabel("sequence length $T$")
ax.set_ylabel("seconds (one head)")
ax.legend(frameon=False)
plotlib.save(f, "t2_cost")